In [ ]:
!pwd

In [ ]:
%matplotlib inline

# attempt_25.ipynb -- Diabetic Retinopathy Detection

**Changes vs attempt_24.ipynb:**

- **(C-8) 5-fold cross-validation ensemble of ResNeXt50.**
  Train 5 independent ResNeXt50 models on stratified 80/20 splits of the combined
  train+val pool (2500 images). Final FT prediction = average across 5 folds.
  OOF (out-of-fold) val AUC gives an unbiased estimate with no data wasted.

- **(C-9) Extended 6-pass TTA.**
  Original 4-pass TTA (orig + hflip + vflip + rot180) extended with rot90 and rot270.
  Fundus images are rotationally symmetric so all rotations are valid; more passes
  reduce score variance at no training cost.

## 1. Imports & Setup

In [ ]:
from __future__ import print_function, division
import os, csv
import torch
import pandas as pd
from skimage import io, transform, util, color
from sklearn import metrics
from sklearn.model_selection import StratifiedKFold
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, utils, models
from torchvision.models import densenet121, DenseNet121_Weights
import torchvision.transforms.functional as TF
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim import lr_scheduler
import time
import copy
from PIL import Image
from zipfile import ZipFile
import random
import numpy.random as npr
import cv2
import warnings
import itertools

warnings.filterwarnings('ignore')
random.seed(42)
npr.seed(42)
torch.manual_seed(42)
torch.backends.cudnn.enabled = False

plt.ion()
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(device)

In [ ]:
# CSVs and images/ folder live at the project root
DATA_ROOT   = '/kaggle/input/datasets/mariamuozperez/lab5-cv'

# Directory containing pre-trained .pth files (not used for fold training)
MODELS_PATH = '/kaggle/working'

# Directory for new .pth checkpoints and submission outputs
SAVE_PATH   = '/kaggle/working'

TRAIN_BATCH = 32    # training batches at 380px
VAL_BATCH   = 128   # val/test scoring at 380px
N_FOLDS     = 5     # number of CV folds

In [ ]:
# Run once to extract data, then comment out
# import zipfile
# with zipfile.ZipFile('./db.zip', 'r') as z:
#     z.extractall('./data')

## 2. Dataset

In [ ]:
class RetinopathyDataset(Dataset):
    """Retinopathy dataset."""

    def __init__(self, csv_file, root_dir, transform=None, maxSize=0):
        self.dataset = pd.read_csv(csv_file, header=0,
                                   dtype={'id': str, 'eye': int, 'label': int})
        if maxSize > 0:
            idx = np.random.RandomState(seed=42).permutation(range(len(self.dataset)))
            self.dataset = self.dataset.iloc[idx[:maxSize]].reset_index(drop=True)
        self.root_dir = root_dir
        self.img_dir  = os.path.join(root_dir, 'images')
        self.transform = transform
        self.levels  = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative DR']
        self.classes = ['No DR', 'DR']

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()
        img_name = os.path.join(self.img_dir, self.dataset.id[idx] + '.jpg')
        image = io.imread(img_name)
        if self.dataset.eye[idx] == 1:
            image = image[:, ::-1, :]
        sample = {
            'image': image,
            'eye':   self.dataset.eye[idx],
            'label': (self.dataset.label[idx] > 0).astype(dtype=np.int64)
        }
        if self.transform:
            sample = self.transform(sample)
        return sample


class FoldDataset(Dataset):
    """Dataset built from a DataFrame slice (used for per-fold train/val splits)."""

    def __init__(self, df, root_dir, transform=None):
        self.dataset  = df.reset_index(drop=True)
        self.root_dir = root_dir
        self.img_dir  = os.path.join(root_dir, 'images')
        self.transform = transform
        self.classes  = ['No DR', 'DR']

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()
        row = self.dataset.iloc[idx]
        img_name = os.path.join(self.img_dir, row['id'] + '.jpg')
        image = io.imread(img_name)
        if row['eye'] == 1:
            image = image[:, ::-1, :]
        sample = {
            'image': image,
            'eye':   int(row['eye']),
            'label': int(row['label'] > 0),
        }
        if self.transform:
            sample = self.transform(sample)
        return sample

## 3. Transforms

In [ ]:
class CropByEye(object):
    """Threshold-based eye segmentation and crop."""
    def __init__(self, threshold, border):
        self.threshold = threshold
        self.border = (border, border) if isinstance(border, int) else border

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        h, w = image.shape[:2]
        imgray = color.rgb2gray(image)
        _, mask = cv2.threshold(imgray, self.threshold, 1, cv2.THRESH_BINARY)
        sidx = np.nonzero(mask)
        if len(sidx[0]) < 20:
            return {'image': image, 'eye': eye, 'label': label}
        minx = np.maximum(sidx[1].min() - self.border[1], 0)
        maxx = np.minimum(sidx[1].max() + 1 + self.border[1], w)
        miny = np.maximum(sidx[0].min() - self.border[0], 0)
        maxy = np.minimum(sidx[0].max() + 1 + self.border[1], h)
        image = image[miny:maxy, minx:maxx, ...]
        return {'image': image, 'eye': eye, 'label': label}


class BenGraham(object):
    """Local contrast normalisation for fundus images."""
    def __init__(self, sigmaX=10):
        self.sigmaX = sigmaX

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        if image.dtype == np.uint8:
            img_u8 = image
        else:
            img_u8 = (np.clip(image, 0, 1) * 255).astype(np.uint8)
        blurred  = cv2.GaussianBlur(img_u8, (0, 0), self.sigmaX)
        enhanced = cv2.addWeighted(img_u8, 4, blurred, -4, 128)
        enhanced = np.clip(enhanced, 0, 255).astype(np.uint8)
        mask = np.zeros(enhanced.shape, dtype=np.uint8)
        h, w = enhanced.shape[:2]
        cv2.circle(mask, (w // 2, h // 2), int(0.9 * min(h, w) / 2), (1, 1, 1), -1, 8, 0)
        enhanced = enhanced * mask + 128 * (1 - mask)
        return {'image': enhanced.astype(np.float32) / 255.0, 'eye': eye, 'label': label}


class Rescale(object):
    """Rescale image to output_size (int = shortest side; tuple = exact)."""
    def __init__(self, output_size):
        self.output_size = output_size

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        h, w = image.shape[:2]
        if isinstance(self.output_size, int):
            new_h = self.output_size * h / w if h > w else self.output_size
            new_w = self.output_size if h > w else self.output_size * w / h
        else:
            new_h, new_w = self.output_size
        image = transform.resize(image, (int(new_h), int(new_w)))
        return {'image': image, 'eye': eye, 'label': label}


class RandomCrop(object):
    """Random crop to output_size."""
    def __init__(self, output_size):
        self.output_size = (output_size, output_size) if isinstance(output_size, int) else output_size

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        h, w = image.shape[:2]
        new_h, new_w = self.output_size
        top  = np.random.randint(0, h - new_h) if h > new_h else 0
        left = np.random.randint(0, w - new_w) if w > new_w else 0
        image = image[top:top + new_h, left:left + new_w]
        return {'image': image, 'eye': eye, 'label': label}


class CenterCrop(object):
    """Centre crop to output_size."""
    def __init__(self, output_size):
        self.output_size = (output_size, output_size) if isinstance(output_size, int) else output_size

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        h, w = image.shape[:2]
        new_h, new_w = self.output_size
        top  = int((h - new_h) / 2) if h > new_h else 0
        left = int((w - new_w) / 2) if w > new_w else 0
        image = image[top:top + new_h, left:left + new_w]
        return {'image': image, 'eye': eye, 'label': label}


class ToTensor(object):
    """Convert HxWxC numpy array to CxHxW torch tensor."""
    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        image = torch.from_numpy(image.transpose((2, 0, 1)))
        label = torch.tensor(label, dtype=torch.long)
        return {'image': image, 'eye': eye, 'label': label}


class Normalize(object):
    """Normalise per-channel using mean and std."""
    def __init__(self, mean, std):
        self.mean = np.array(mean)
        self.std  = np.array(std)

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        dtype = image.dtype
        mean = torch.as_tensor(self.mean, dtype=dtype, device=image.device)
        std  = torch.as_tensor(self.std,  dtype=dtype, device=image.device)
        image.sub_(mean[:, None, None]).div_(std[:, None, None])
        return {'image': image, 'eye': eye, 'label': label}


class TVCenterCrop(object):
    def __init__(self, size):
        self.CC = transforms.CenterCrop(size)

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        pil = Image.fromarray(util.img_as_ubyte(image))
        image = util.img_as_float(np.asarray(self.CC(pil)))
        return {'image': image, 'eye': eye, 'label': label}


class TVRandomHorizontalFlip(object):
    def __init__(self, p=0.5):
        self.flip = transforms.RandomHorizontalFlip(p=p)

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        pil = Image.fromarray(util.img_as_ubyte(image))
        image = util.img_as_float(np.asarray(self.flip(pil)))
        return {'image': image, 'eye': eye, 'label': label}


class TVRandomRotation(object):
    def __init__(self, degrees=15):
        self.rotate = transforms.RandomRotation(degrees=degrees)

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        pil = Image.fromarray(util.img_as_ubyte(image))
        image = util.img_as_float(np.asarray(self.rotate(pil)))
        return {'image': image, 'eye': eye, 'label': label}


class TVColorJitter(object):
    def __init__(self, brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05):
        self.jitter = transforms.ColorJitter(
            brightness=brightness, contrast=contrast,
            saturation=saturation, hue=hue)

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        pil = Image.fromarray(util.img_as_ubyte(image))
        image = util.img_as_float(np.asarray(self.jitter(pil)))
        return {'image': image, 'eye': eye, 'label': label}

## 4. Data Pipelines

In [ ]:
pixel_mean = [0.485, 0.456, 0.406]
pixel_std  = [0.229, 0.224, 0.225]

# Training: augmentation pipeline — 380 px, full 360° rotation
train_transform = transforms.Compose([
    CropByEye(0.10, 1),
    BenGraham(sigmaX=10),
    Rescale(416),
    TVRandomHorizontalFlip(p=0.5),
    TVRandomRotation(degrees=180),
    TVColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    RandomCrop(380),
    ToTensor(),
    Normalize(mean=pixel_mean, std=pixel_std),
])

# Val / test: Rescale(416) then CenterCrop(380)
eval_transform = transforms.Compose([
    CropByEye(0.10, 1),
    BenGraham(sigmaX=10),
    Rescale(416),
    CenterCrop(380),
    ToTensor(),
    Normalize(mean=pixel_mean, std=pixel_std),
])

# Fixed train/val datasets for CustomNetV2 (unchanged from attempt_24)
train_dataset = RetinopathyDataset(
    csv_file=os.path.join(DATA_ROOT, 'train.csv'),
    root_dir=DATA_ROOT,
    maxSize=0,
    transform=train_transform)

val_dataset = RetinopathyDataset(
    csv_file=os.path.join(DATA_ROOT, 'val.csv'),
    root_dir=DATA_ROOT,
    transform=eval_transform)

test_dataset = RetinopathyDataset(
    csv_file=os.path.join(DATA_ROOT, 'test.csv'),
    root_dir=DATA_ROOT,
    transform=eval_transform)

print(f'Train: {len(train_dataset)}  Val: {len(val_dataset)}  Test: {len(test_dataset)}')

In [ ]:
# Sanity check: verify BenGraham is not producing all-gray images
# Expected: mean near 0 (after ImageNet normalization)
_sample = train_dataset[0]
_img = _sample['image']
print(f'Image shape: {_img.shape}  dtype: {_img.dtype}  '
      f'mean: {_img.float().mean():.3f}  std: {_img.float().std():.3f}')
del _sample, _img

In [ ]:
# Class-balanced sampler for CustomNetV2
train_labels_bin_for_sampler = (train_dataset.dataset['label'].values > 0).astype(int)
class_counts  = np.bincount(train_labels_bin_for_sampler)
sample_weights = np.where(train_labels_bin_for_sampler == 1,
                           1.0 / class_counts[1],
                           1.0 / class_counts[0])
sampler = WeightedRandomSampler(
    weights=torch.tensor(sample_weights, dtype=torch.float),
    num_samples=len(train_dataset),
    replacement=True,
)

train_dataloader = DataLoader(train_dataset, batch_size=TRAIN_BATCH, sampler=sampler,  num_workers=0)
val_dataloader   = DataLoader(val_dataset,   batch_size=VAL_BATCH,   shuffle=False, num_workers=0)
test_dataloader  = DataLoader(test_dataset,  batch_size=VAL_BATCH,   shuffle=False, num_workers=0)

# Class-weighted loss
train_labels_bin = (train_dataset.dataset['label'].values > 0).astype(int)
n_neg = int((train_labels_bin == 0).sum())
n_pos = int((train_labels_bin == 1).sum())
pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float).to(device)
print(f'No-DR: {n_neg}  DR: {n_pos}  pos_weight: {pos_weight.item():.3f}')

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# Globals used by train_model — will be overwritten per fold for FT training
dataloaders  = {'train': train_dataloader, 'val': val_dataloader}
dataset_sizes = {'train': len(train_dataset), 'val': len(val_dataset)}
class_names  = train_dataset.classes

## 5. Training & Evaluation Utilities

In [ ]:
def train_model(model, criterion, optimizer, scheduler, num_epochs=25, patience=7, label_smoothing=0.0):
    """Train with early stopping on val AUC. Returns best-weight model."""
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_auc   = 0.0
    best_epoch = -1
    no_improve = 0

    for epoch in range(num_epochs):
        print('Epoch {}/{}'.format(epoch, num_epochs - 1))
        print('-' * 10)

        for phase in ['train', 'val']:
            model.train() if phase == 'train' else model.eval()

            numSamples   = dataset_sizes[phase]
            outputs_m    = np.zeros((numSamples,), dtype=float)
            labels_m     = np.zeros((numSamples,), dtype=int)
            running_loss = 0.0
            contSamples  = 0

            for sample in dataloaders[phase]:
                inputs = sample['image'].to(device).float()
                labels = sample['label'].to(device).float()
                batchSize = labels.shape[0]
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    logits = model(inputs).flatten()
                    if label_smoothing > 0.0 and phase == 'train':
                        labels_ls = labels * (1 - label_smoothing) + label_smoothing / 2.0
                        loss = criterion(logits, labels_ls)
                    else:
                        loss = criterion(logits, labels)
                    scores = torch.sigmoid(logits).detach()
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * batchSize
                outputs_m[contSamples:contSamples + batchSize] = scores.cpu().numpy()
                labels_m [contSamples:contSamples + batchSize] = labels.cpu().numpy()
                contSamples += batchSize

            if phase == 'train':
                scheduler.step()

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_auc  = metrics.roc_auc_score(labels_m, outputs_m)
            print('{} Loss: {:.4f}  AUC: {:.4f}'.format(phase, epoch_loss, epoch_auc))

            if phase == 'val':
                if epoch_auc > best_auc:
                    best_auc       = epoch_auc
                    best_epoch     = epoch
                    best_model_wts = copy.deepcopy(model.state_dict())
                    no_improve     = 0
                else:
                    no_improve += 1
                    if no_improve >= patience:
                        print(f'Early stopping: no improvement for {patience} epochs.')
                        model.load_state_dict(best_model_wts)
                        return model
        print()

    elapsed = time.time() - since
    print('Training complete in {:.0f}m {:.0f}s'.format(elapsed // 60, elapsed % 60))
    print('Best model: epoch {:d}  val AUC: {:.4f}'.format(best_epoch, best_auc))
    model.load_state_dict(best_model_wts)
    return model

In [ ]:
def eval_val_auc(model, name, tta=False):
    """Run model on fixed val set. tta=True: 4-pass TTA (unchanged from attempt_24)."""
    model.eval()
    n        = len(val_dataset)
    scores_m = np.zeros((n, 1), dtype=float)
    labels_m = np.zeros((n,),   dtype=int)
    cont = 0
    with torch.no_grad():
        for sample in val_dataloader:
            inputs = sample['image'].to(device).float()
            bs = inputs.shape[0]
            if tta:
                s1  = torch.sigmoid(model(inputs))
                s2  = torch.sigmoid(model(torch.flip(inputs, dims=[3])))
                s3  = torch.sigmoid(model(torch.flip(inputs, dims=[2])))
                s4  = torch.sigmoid(model(torch.flip(inputs, dims=[2, 3])))
                out = (s1 + s2 + s3 + s4) / 4.0
            else:
                out = torch.sigmoid(model(inputs))
            scores_m[cont:cont + bs, :] = out.cpu().numpy()
            labels_m[cont:cont + bs]    = sample['label'].numpy()
            cont += bs
    auc    = metrics.roc_auc_score(labels_m, scores_m)
    suffix = ' (TTA-4)' if tta else ''
    print(f'{name}{suffix}  --  val AUC: {auc:.4f}')
    return auc


def test_model(model, tta=False):
    """Run model on test set. Returns (1000, 1) score array."""
    model.eval()
    numSamples = len(test_dataset)
    outputs_m  = np.zeros((numSamples, 1), dtype=float)
    contSamples = 0
    with torch.no_grad():
        for sample in test_dataloader:
            inputs = sample['image'].to(device).float()
            bs = inputs.shape[0]
            if tta:
                s1  = torch.sigmoid(model(inputs))
                s2  = torch.sigmoid(model(torch.flip(inputs, dims=[3])))
                s3  = torch.sigmoid(model(torch.flip(inputs, dims=[2])))
                s4  = torch.sigmoid(model(torch.flip(inputs, dims=[2, 3])))
                out = (s1 + s2 + s3 + s4) / 4.0
            else:
                out = torch.sigmoid(model(inputs))
            outputs_m[contSamples:contSamples + bs, :] = out.cpu().numpy()
            contSamples += bs
    return outputs_m

---
## 6. CustomNetV2 (CUSTOM category)

**Key improvements over old CustomNet:**
- 3×3 same-padding convs (vs 5×5 no-padding)
- BatchNorm in every conv block
- SE (Squeeze-and-Excitation) blocks in the 3 deepest stages
- Global Average Pooling instead of Flatten
- AdamW + CosineAnnealingLR

In [ ]:
class SEBlock(nn.Module):
    """Squeeze-and-Excitation block (Hu et al., 2017)."""
    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden = max(channels // reduction, 4)
        self.squeeze = nn.AdaptiveAvgPool2d(1)
        self.excitation = nn.Sequential(
            nn.Linear(channels, hidden, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, channels, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, x):
        b, c, _, _ = x.shape
        w = self.squeeze(x).view(b, c)
        w = self.excitation(w).view(b, c, 1, 1)
        return x * w


class CustomNetV2(nn.Module):
    """5-block CNN with 3x3 same-convolutions, BatchNorm, GAP, SE blocks."""
    def __init__(self, se_reduction=16):
        super().__init__()

        def _block(cin, cout, use_se=False):
            layers = [
                nn.Conv2d(cin, cout, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(cout),
                nn.ReLU(inplace=True),
            ]
            if use_se:
                layers.append(SEBlock(cout, reduction=se_reduction))
            layers.append(nn.MaxPool2d(2, 2))
            return nn.Sequential(*layers)

        self.features = nn.Sequential(
            _block(3,   32,  use_se=False),
            _block(32,  64,  use_se=False),
            _block(64,  128, use_se=True ),
            _block(128, 256, use_se=True ),
            _block(256, 256, use_se=True ),
        )
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.2),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.4),
            nn.Linear(128, 1),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.gap(x).flatten(1)
        return self.classifier(x)

In [ ]:
_net = CustomNetV2().to(device)
_inp = next(iter(train_dataloader))['image'].to(device).float()
with torch.no_grad():
    _out = _net(_inp)
print(f'Input shape:  {_inp.shape}')
print(f'Output shape: {_out.shape}')
total = sum(p.numel() for p in _net.parameters())
print(f'CustomNetV2 total params: {total:,}')
del _net, _inp, _out

In [ ]:
random.seed(42); npr.seed(42); torch.manual_seed(42); torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

customNetV2 = CustomNetV2().to(device)
optimizer_custom = optim.AdamW(customNetV2.parameters(), lr=1e-3, weight_decay=5e-3)
scheduler_custom = lr_scheduler.CosineAnnealingLR(optimizer_custom, T_max=50)

In [ ]:
customNetV2 = train_model(customNetV2, criterion, optimizer_custom, scheduler_custom,
                          num_epochs=50, patience=10)

In [ ]:
torch.save(customNetV2.state_dict(), os.path.join(SAVE_PATH, 'best_customnetv2_se.pth'))
print(f'Saved: {os.path.join(SAVE_PATH, "best_customnetv2_se.pth")}')
auc_custom = eval_val_auc(customNetV2, 'CustomNetV2 + SE')

# Fine-tuning category — 5-fold CV ResNeXt50

In [ ]:
!pip install timm

In [ ]:
from torchvision.models import (
    resnext50_32x4d, ResNeXt50_32X4D_Weights,
)
import timm

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    npr.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
def build_resnext50(device):
    """Build ResNeXt50 with custom head and return model + unfreezing/group fns."""
    model = resnext50_32x4d(weights=ResNeXt50_32X4D_Weights.IMAGENET1K_V2)
    for p in model.parameters():
        p.requires_grad = False
    in_feats = model.fc.in_features
    model.fc = nn.Sequential(nn.Dropout(p=0.5), nn.Linear(in_feats, 1))
    model = model.to(device)

    def stage1_unfreeze(m):
        for p in m.layer4.parameters(): p.requires_grad = True
        for p in m.fc.parameters():     p.requires_grad = True

    def stage2_unfreeze(m):
        for p in m.layer3.parameters(): p.requires_grad = True

    def stage1_groups(m):
        return [
            {'params': m.layer4.parameters(), 'lr': 3e-4},
            {'params': m.fc.parameters(),     'lr': 3e-4},
        ]

    def stage2_groups(m):
        return [
            {'params': m.layer3.parameters(), 'lr': 2e-5},
            {'params': m.layer4.parameters(), 'lr': 3e-5},
            {'params': m.fc.parameters(),     'lr': 1e-4},
        ]

    return model, stage1_unfreeze, stage2_unfreeze, stage1_groups, stage2_groups

In [ ]:
def get_val_scores(model, loader, device, tta=False):
    """Score a dataloader. tta=True uses 6-pass TTA (orig+hflip+vflip+rot180+rot90+rot270)."""
    model.eval()
    n = len(loader.dataset)
    scores = np.zeros((n, 1), dtype=np.float32)
    labels = np.zeros((n,),   dtype=np.int64)
    cont = 0
    with torch.no_grad():
        for sample in loader:
            inputs = sample['image'].to(device).float()
            bs = inputs.shape[0]
            if tta:
                s1 = torch.sigmoid(model(inputs))
                s2 = torch.sigmoid(model(torch.flip(inputs, dims=[3])))
                s3 = torch.sigmoid(model(torch.flip(inputs, dims=[2])))
                s4 = torch.sigmoid(model(torch.flip(inputs, dims=[2, 3])))
                s5 = torch.sigmoid(model(torch.rot90(inputs, 1, [2, 3])))
                s6 = torch.sigmoid(model(torch.rot90(inputs, 3, [2, 3])))
                out = (s1 + s2 + s3 + s4 + s5 + s6) / 6.0
            else:
                out = torch.sigmoid(model(inputs))
            scores[cont:cont+bs, :] = out.detach().cpu().numpy()
            labels[cont:cont+bs]    = sample['label'].numpy()
            cont += bs
    return scores, labels


def get_test_scores(model, loader, device, tta=True):
    """Score test set. Returns (N, 1) array. tta=True uses 6-pass TTA."""
    model.eval()
    n = len(loader.dataset)
    scores = np.zeros((n, 1), dtype=np.float32)
    cont = 0
    with torch.no_grad():
        for sample in loader:
            inputs = sample['image'].to(device).float()
            bs = inputs.shape[0]
            if tta:
                s1 = torch.sigmoid(model(inputs))
                s2 = torch.sigmoid(model(torch.flip(inputs, dims=[3])))
                s3 = torch.sigmoid(model(torch.flip(inputs, dims=[2])))
                s4 = torch.sigmoid(model(torch.flip(inputs, dims=[2, 3])))
                s5 = torch.sigmoid(model(torch.rot90(inputs, 1, [2, 3])))
                s6 = torch.sigmoid(model(torch.rot90(inputs, 3, [2, 3])))
                out = (s1 + s2 + s3 + s4 + s5 + s6) / 6.0
            else:
                out = torch.sigmoid(model(inputs))
            scores[cont:cont+bs, :] = out.detach().cpu().numpy()
            cont += bs
    return scores

## 7. Combined Train+Val Dataset for 5-fold CV

In [ ]:
train_df = pd.read_csv(os.path.join(DATA_ROOT, 'train.csv'),
                      dtype={'id': str, 'eye': int, 'label': int})
val_df   = pd.read_csv(os.path.join(DATA_ROOT, 'val.csv'),
                       dtype={'id': str, 'eye': int, 'label': int})
combined_df = pd.concat([train_df, val_df], ignore_index=True)

combined_labels_bin = (combined_df['label'].values > 0).astype(int)
n_neg_c = int((combined_labels_bin == 0).sum())
n_pos_c = int((combined_labels_bin == 1).sum())
print(f'Combined: {len(combined_df)} images — No-DR: {n_neg_c}  DR: {n_pos_c}')

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
fold_splits = list(skf.split(combined_df, combined_labels_bin))
for i, (tr, va) in enumerate(fold_splits):
    print(f'Fold {i+1}: train={len(tr)}  val={len(va)}  '
          f'DR_train={combined_labels_bin[tr].sum()}  DR_val={combined_labels_bin[va].sum()}')

## 8. K-fold Training Loop

In [ ]:
fold_val_scores_oof = np.zeros((len(combined_df), 1), dtype=np.float32)
fold_val_labels_oof = np.zeros(len(combined_df), dtype=np.int64)
fold_test_scores_list = []

for fold_idx, (train_idx, val_idx) in enumerate(fold_splits):
    print(f'\n{"="*70}')
    print(f'FOLD {fold_idx+1}/{N_FOLDS}')
    print(f'{"="*70}')

    fold_train_df = combined_df.iloc[train_idx]
    fold_val_df   = combined_df.iloc[val_idx]

    fold_train_ds = FoldDataset(fold_train_df, DATA_ROOT, transform=train_transform)
    fold_val_ds   = FoldDataset(fold_val_df,   DATA_ROOT, transform=eval_transform)

    fold_train_lbl = (fold_train_df['label'].values > 0).astype(int)
    fold_cc = np.bincount(fold_train_lbl)
    fold_sw = np.where(fold_train_lbl == 1, 1.0 / fold_cc[1], 1.0 / fold_cc[0])
    fold_sampler = WeightedRandomSampler(
        weights=torch.tensor(fold_sw, dtype=torch.float),
        num_samples=len(fold_train_ds), replacement=True)

    fold_train_loader = DataLoader(fold_train_ds, batch_size=TRAIN_BATCH,
                                   sampler=fold_sampler, num_workers=0)
    fold_val_loader   = DataLoader(fold_val_ds,   batch_size=VAL_BATCH,
                                   shuffle=False, num_workers=0)

    fold_n_pos = int(fold_train_lbl.sum())
    fold_n_neg = len(fold_train_lbl) - fold_n_pos
    fold_pw = torch.tensor([fold_n_neg / fold_n_pos], dtype=torch.float).to(device)
    fold_criterion = nn.BCEWithLogitsLoss(pos_weight=fold_pw)

    # Point the global dataloaders/dataset_sizes to this fold so train_model works
    dataloaders['train']   = fold_train_loader
    dataloaders['val']     = fold_val_loader
    dataset_sizes['train'] = len(fold_train_ds)
    dataset_sizes['val']   = len(fold_val_ds)

    # --- Stage 1 ---
    set_seed(42 + fold_idx)
    model, s1_unfreeze, s2_unfreeze, s1_groups, s2_groups = build_resnext50(device)
    s1_unfreeze(model)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'Stage 1 trainable params: {trainable:,}')

    opt_s1  = optim.AdamW(s1_groups(model), weight_decay=1e-2)
    sched_s1 = lr_scheduler.CosineAnnealingLR(opt_s1, T_max=30)
    model = train_model(model, fold_criterion, opt_s1, sched_s1,
                        num_epochs=30, patience=7, label_smoothing=0.05)

    s1_path = os.path.join(SAVE_PATH, f'best_resnext50_fold{fold_idx}_s1.pth')
    torch.save(model.state_dict(), s1_path)
    print(f'Saved: {s1_path}')

    # Stage 1 TTA val AUC
    model_gpu = model
    s1_scores, s1_labels = get_val_scores(model_gpu, fold_val_loader, device, tta=True)
    auc_s1_tta = metrics.roc_auc_score(s1_labels, s1_scores)
    print(f'Fold {fold_idx+1} Stage 1 TTA-6 val AUC: {auc_s1_tta:.4f}')

    best_model = copy.deepcopy(model)
    best_auc   = auc_s1_tta
    best_scores = s1_scores

    # --- Stage 2 ---
    if auc_s1_tta >= 0.752:
        print(f'Fold {fold_idx+1}: running Stage 2...')
        model_s2, s1_unfreeze2, s2_unfreeze2, _, s2_groups2 = build_resnext50(device)
        model_s2.load_state_dict(torch.load(s1_path, map_location=device))
        s1_unfreeze2(model_s2)
        s2_unfreeze2(model_s2)

        trainable_s2 = sum(p.numel() for p in model_s2.parameters() if p.requires_grad)
        print(f'Stage 2 trainable params: {trainable_s2:,}')

        opt_s2 = optim.AdamW(s2_groups2(model_s2), weight_decay=1e-2)
        warmup = lr_scheduler.LinearLR(opt_s2, start_factor=0.1, end_factor=1.0, total_iters=3)
        cosine = lr_scheduler.CosineAnnealingLR(opt_s2, T_max=12)
        sched_s2 = lr_scheduler.SequentialLR(opt_s2, schedulers=[warmup, cosine], milestones=[3])

        model_s2 = train_model(model_s2, fold_criterion, opt_s2, sched_s2,
                               num_epochs=15, patience=5, label_smoothing=0.05)

        s2_scores, _ = get_val_scores(model_s2, fold_val_loader, device, tta=True)
        auc_s2_tta   = metrics.roc_auc_score(s1_labels, s2_scores)
        print(f'Fold {fold_idx+1} Stage 2 TTA-6 val AUC: {auc_s2_tta:.4f}')

        if auc_s2_tta > auc_s1_tta:
            s2_path = os.path.join(SAVE_PATH, f'best_resnext50_fold{fold_idx}_s2.pth')
            torch.save(model_s2.state_dict(), s2_path)
            print(f'Fold {fold_idx+1}: Stage 2 better, saved {s2_path}')
            best_model  = copy.deepcopy(model_s2)
            best_auc    = auc_s2_tta
            best_scores = s2_scores
            del model_s2
        else:
            print(f'Fold {fold_idx+1}: Stage 2 did not improve, keeping Stage 1')
            del model_s2
    else:
        print(f'Fold {fold_idx+1}: Stage 1 AUC {auc_s1_tta:.4f} < gate 0.752, skipping Stage 2')

    # Store OOF val predictions
    fold_val_scores_oof[val_idx] = best_scores
    fold_val_labels_oof[val_idx] = s1_labels

    # Test predictions for this fold
    best_model_gpu = best_model.to(device)
    fold_test = get_test_scores(best_model_gpu, test_dataloader, device, tta=True)
    fold_test_scores_list.append(fold_test)

    best_model_gpu.cpu()
    del best_model, model
    torch.cuda.empty_cache()
    print(f'Fold {fold_idx+1} complete — best TTA-6 val AUC: {best_auc:.4f}')

In [ ]:
# Out-of-fold AUC: each image was val exactly once, giving an unbiased estimate
oof_auc = metrics.roc_auc_score(fold_val_labels_oof, fold_val_scores_oof)
print(f'\nOOF val AUC (TTA-6, 5 folds): {oof_auc:.4f}')
print(f'(attempt_24 best single-split val AUC was 0.8671)')

## 9. Average Fold Test Scores

In [ ]:
# Average predictions across all 5 fold models
best_ft_test_scores = np.mean(fold_test_scores_list, axis=0)
print(f'Averaged test scores shape: {best_ft_test_scores.shape}')

np.savetxt(os.path.join(SAVE_PATH, 'ft_kfold_submission.csv'),
           best_ft_test_scores, fmt='%.8f')
print(f'Saved: {os.path.join(SAVE_PATH, "ft_kfold_submission.csv")}')

---
## 10. Generate Test Outputs & Submit

In [ ]:
print('=== Final validation AUC check ===')
auc_custom_final = eval_val_auc(customNetV2, 'CustomNetV2 + SE')
print(f'FT OOF val AUC (TTA-6, 5-fold): {oof_auc:.4f}')

In [ ]:
outputs_custom = test_model(customNetV2, tta=False)
outputs_ft     = best_ft_test_scores

assert outputs_custom.shape == (1000, 1), f'Expected (1000,1), got {outputs_custom.shape}'
assert outputs_ft.shape     == (1000, 1), f'Expected (1000,1), got {outputs_ft.shape}'
assert np.isfinite(outputs_custom).all(), 'NaN/inf in custom scores'
assert np.isfinite(outputs_ft).all(),     'NaN/inf in ft scores'
print('Shapes and finite-value checks passed.')

custom_csv = os.path.join(SAVE_PATH, 'output_custom.csv')
ft_csv     = os.path.join(SAVE_PATH, 'output_ft.csv')
zip_path   = os.path.join(SAVE_PATH, 'codabench_submission.zip')

with open(custom_csv, mode='w', newline='') as f:
    csv.writer(f).writerows(outputs_custom)
with open(ft_csv, mode='w', newline='') as f:
    csv.writer(f).writerows(outputs_ft)

with ZipFile(zip_path, 'w') as zf:
    zf.write(custom_csv, 'output_custom.csv')
    zf.write(ft_csv,     'output_ft.csv')

print(f'Created: {zip_path}')
print(f'CUSTOM val AUC: {auc_custom_final:.4f}')
print(f'FT     OOF AUC: {oof_auc:.4f}')